# Resume Matching Pipeline for TalentLens

This notebook implements a semantic search system for matching job descriptions to student resumes.

## System Architecture

```
Job Description → Embedding → Vector Search → Ranked Resumes
                                    ↓
                           Member Resume DB
                           (Pre-embedded)
```

## Data Flow

1. **Training Data**: Reddit + Discord + Kaggle resumes
2. **Production Data**: DS3 member resumes
3. **Query**: Job description or skill keywords
4. **Output**: Ranked list of member resumes with similarity scores

## Phase 1: Setup and Data Loading

In [3]:
# Install required packages (run once)
!pip install sentence-transformers
!pip install pdfplumber
!pip install pytesseract
!pip install Pillow
!pip install python-docx
!pip install spacy
!pip install faiss-gpu  # or faiss-gpu if you have GPU
!pip install numpy
!pip install pandas

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports successful!")

  Using cached torch-2.10.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (31 kB)
  Using cached numpy-2.4.2-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.17.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached filelock-3.20.3-py3-none-any.whl.metadata (2.1 kB)
  Using cached hf_xet-1.2.0-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached certifi-2026.1.4-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.w

## Phase 2: Resume Text Extraction

Extract text from all resume formats (PDF, images, DOCX)

In [4]:
import pdfplumber
from PIL import Image
import pytesseract
from docx import Document

def extract_text_from_pdf(file_path):
    """Extract text from PDF using pdfplumber"""
    try:
        with pdfplumber.open(file_path) as pdf:
            text = ''
            for page in pdf.pages:
                text += page.extract_text() or ''
            return text.strip()
    except Exception as e:
        print(f"Error extracting PDF {file_path}: {e}")
        return ""

def extract_text_from_image(file_path):
    """Extract text from image using OCR"""
    try:
        image = Image.open(file_path)
        text = pytesseract.image_to_string(image)
        return text.strip()
    except Exception as e:
        print(f"Error extracting image {file_path}: {e}")
        return ""

def extract_text_from_docx(file_path):
    """Extract text from DOCX"""
    try:
        doc = Document(file_path)
        text = '\n'.join([para.text for para in doc.paragraphs])
        return text.strip()
    except Exception as e:
        print(f"Error extracting DOCX {file_path}: {e}")
        return ""

def extract_resume_text(file_path):
    """Extract text from resume file based on extension"""
    file_path = Path(file_path)
    ext = file_path.suffix.lower()
    
    if ext == '.pdf':
        return extract_text_from_pdf(file_path)
    elif ext in ['.jpg', '.jpeg', '.png', '.gif']:
        return extract_text_from_image(file_path)
    elif ext in ['.docx', '.doc']:
        return extract_text_from_docx(file_path)
    elif ext == '.txt':
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read().strip()
    else:
        print(f"Unsupported file type: {ext}")
        return ""

print("✓ Text extraction functions loaded!")

✓ Text extraction functions loaded!


In [5]:
# Process all resumes from different sources
def process_resume_folder(folder_path, source_name, limit=None):
    """Process all resumes in a folder and extract text"""
    folder = Path(folder_path)
    resumes = []
    
    if not folder.exists():
        print(f"Folder not found: {folder}")
        return resumes
    
    files = list(folder.glob('*'))
    if limit:
        files = files[:limit]
    
    print(f"Processing {len(files)} files from {source_name}...")
    
    for file_path in files:
        if file_path.is_file() and not file_path.name.startswith('.'):
            text = extract_resume_text(file_path)
            
            if text and len(text) > 100:  # Only keep if we got meaningful text
                resumes.append({
                    'filename': file_path.name,
                    'source': source_name,
                    'text': text,
                    'file_path': str(file_path)
                })
    
    print(f"✓ Extracted text from {len(resumes)} resumes")
    return resumes

# Process all datasets
all_resumes = []

# 1. DS3 Member Resumes (PRIMARY - these are what we'll search)
member_resumes = process_resume_folder(
    'data/ds3/member_resumes',
    source_name='ds3_members'
)
all_resumes.extend(member_resumes)

# 2. Reddit Resumes (for training/reference)
reddit_resumes = process_resume_folder(
    'data/reddit/resumes/files',
    source_name='reddit',
    limit=100  # Limit to avoid processing too many
)
all_resumes.extend(reddit_resumes)

# 3. Kaggle Dataset (if you have it)
# kaggle_resumes = process_resume_folder('data/resume-dataset/...', 'kaggle', limit=200)
# all_resumes.extend(kaggle_resumes)

print(f"\n{'='*60}")
print(f"Total resumes processed: {len(all_resumes)}")
print(f"  - DS3 Members: {len([r for r in all_resumes if r['source'] == 'ds3_members'])}")
print(f"  - Reddit: {len([r for r in all_resumes if r['source'] == 'reddit'])}")
print(f"{'='*60}")

# Create DataFrame
resumes_df = pd.DataFrame(all_resumes)
resumes_df.head()

Processing 1 files from ds3_members...
✓ Extracted text from 1 resumes
Processing 18 files from reddit...
Error extracting image data/reddit/resumes/files/1kb39fw_direct.jpeg: tesseract is not installed or it's not in your PATH. See README file for more information.
Error extracting image data/reddit/resumes/files/direct_54184671.png: tesseract is not installed or it's not in your PATH. See README file for more information.
Error extracting image data/reddit/resumes/files/Embedded Software Engineer Resume.png: tesseract is not installed or it's not in your PATH. See README file for more information.
Error extracting image data/reddit/resumes/files/1ngk3n2_gallery_1.png: tesseract is not installed or it's not in your PATH. See README file for more information.
Error extracting image data/reddit/resumes/files/Imgur Image.jpeg: tesseract is not installed or it's not in your PATH. See README file for more information.
Error extracting image data/reddit/resumes/files/1kf5zzq_direct.jpeg: te

,filename,source,text,file_path
0,member.txt,ds3_members,"[\n {\n ""full_name"": ""Alan Adams"",\n ""e...",data/ds3/member_resumes/member.txt
1,gdrive_1ZA6kmbFYmR6_aybZ7FQpkL4j8p1pEtF6.pdf,reddit,Aritro Shome\n+91-9073989733 | aritro.shome.of...,data/reddit/resumes/files/gdrive_1ZA6kmbFYmR6_...


## Phase 3: Create Embeddings

Use sentence-transformers to create semantic embeddings

In [6]:
from sentence_transformers import SentenceTransformer
import torch

# Load pre-trained model
# Options:
# - 'all-MiniLM-L6-v2' (fast, small, good for starting)
# - 'all-mpnet-base-v2' (slower, better quality)
# - 'multi-qa-mpnet-base-dot-v1' (optimized for Q&A/search)

print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✓ Model loaded!")

# Create embeddings for all resumes
print("\nCreating embeddings for all resumes...")
print("This may take a few minutes...")

resume_texts = resumes_df['text'].tolist()
embeddings = model.encode(resume_texts, show_progress_bar=True, batch_size=32)

print(f"✓ Created {len(embeddings)} embeddings")
print(f"  Embedding dimension: {embeddings.shape[1]}")

# Add embeddings to dataframe
resumes_df['embedding'] = list(embeddings)

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2404.58it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded!

Creating embeddings for all resumes...
This may take a few minutes...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.26s/it]

✓ Created 2 embeddings
  Embedding dimension: 384


## Phase 4: Build Vector Search Index

Use FAISS for fast similarity search

In [11]:
!pip install faiss-cpu

import faiss

# Separate member resumes from training data
member_df = resumes_df[resumes_df['source'] == 'ds3_members'].copy().reset_index(drop=True)
member_embeddings = np.array(member_df['embedding'].tolist()).astype('float32')

print(f"Building search index for {len(member_df)} member resumes...")

# Create FAISS index
dimension = member_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product (cosine similarity)

# Normalize vectors for cosine similarity
faiss.normalize_L2(member_embeddings)
index.add(member_embeddings)

print(f"✓ Index built with {index.ntotal} resumes")
print(f"  Ready for search!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 17.0 MB/s  0:00:00m0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Building search index for 1 member resumes...
✓ Index built with 1 resumes
  Ready for search!


## Phase 5: Search Function

Main search function for job descriptions

In [12]:
def search_resumes(query, top_k=10, min_score=0.0):
    """
    Search for resumes matching a job description or skills
    
    Args:
        query: Job description or skill keywords
        top_k: Number of results to return
        min_score: Minimum similarity score (0-1)
    
    Returns:
        List of matching resumes with scores
    """
    # Create query embedding
    query_embedding = model.encode([query])
    query_embedding = query_embedding.astype('float32')
    faiss.normalize_L2(query_embedding)
    
    # Search
    scores, indices = index.search(query_embedding, top_k)
    
    # Format results
    results = []
    for i, (idx, score) in enumerate(zip(indices[0], scores[0])):
        if score >= min_score:
            resume = member_df.iloc[idx]
            results.append({
                'rank': i + 1,
                'filename': resume['filename'],
                'score': float(score),
                'file_path': resume['file_path'],
                'text_preview': resume['text'][:300] + '...'
            })
    
    return results

print("✓ Search function ready!")

✓ Search function ready!


## Phase 6: Test the System

Try some example searches

In [13]:
# Example 1: Search by skills
query1 = "Python developer with machine learning and data science experience"
results1 = search_resumes(query1, top_k=5)

print("="*80)
print(f"Query: {query1}")
print("="*80)

for result in results1:
    print(f"\nRank {result['rank']}: {result['filename']}")
    print(f"  Score: {result['score']:.3f}")
    print(f"  Preview: {result['text_preview'][:150]}...")
    print("-"*80)

Query: Python developer with machine learning and data science experience

Rank 1: member.txt
  Score: 0.131
  Preview: [
  {
    "full_name": "Alan Adams",
    "email": "ala036@ucsd.edu",
    "graduation_year": "2029",
    "major": "Data Science (B.S.)",
    "resume_li...
--------------------------------------------------------------------------------


In [14]:
# Example 2: Full job description
job_description = """
Software Engineering Intern

We are looking for a motivated software engineering intern to join our team.

Requirements:
- Strong programming skills in Python, Java, or C++
- Experience with web development (React, Node.js)
- Familiarity with databases (SQL, MongoDB)
- Understanding of data structures and algorithms
- Good communication skills
- Currently pursuing a degree in Computer Science or related field

Preferred:
- Experience with cloud platforms (AWS, GCP, Azure)
- Machine learning or AI experience
- Previous internship experience
"""

results2 = search_resumes(job_description, top_k=10)

print("="*80)
print("Job Description Search Results")
print("="*80)

for result in results2:
    print(f"\n{result['rank']}. {result['filename']}")
    print(f"   Similarity Score: {result['score']:.3f}")

Job Description Search Results

1. member.txt
   Similarity Score: 0.270


## Phase 7: Export for Web Integration

Save the index and metadata for use in your website

In [15]:
# Save the FAISS index
faiss.write_index(index, "resume_index.faiss")
print("✓ FAISS index saved to resume_index.faiss")

# Save member resume metadata
member_df_export = member_df[['filename', 'file_path', 'text']].copy()
member_df_export.to_json('member_resumes_metadata.json', orient='records', indent=2)
print("✓ Metadata saved to member_resumes_metadata.json")

# Save the model name for loading later
config = {
    'model_name': 'all-MiniLM-L6-v2',
    'num_resumes': len(member_df),
    'embedding_dimension': dimension
}
with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)
print("✓ Config saved to config.json")

print("\n" + "="*80)
print("System ready for deployment!")
print("="*80)
print("\nFiles created:")
print("  1. resume_index.faiss - Vector search index")
print("  2. member_resumes_metadata.json - Resume metadata")
print("  3. config.json - System configuration")
print("\nUse these files in your web application backend!")

✓ FAISS index saved to resume_index.faiss
✓ Metadata saved to member_resumes_metadata.json
✓ Config saved to config.json

System ready for deployment!

Files created:
  1. resume_index.faiss - Vector search index
  2. member_resumes_metadata.json - Resume metadata
  3. config.json - System configuration

Use these files in your web application backend!


## Phase 8: Create Simple API (Optional)

Example FastAPI code for your backend

In [16]:
# Save this as api.py for your backend
api_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
import faiss
import json
import numpy as np

app = FastAPI()

# Load model and index on startup
model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index('resume_index.faiss')
with open('member_resumes_metadata.json', 'r') as f:
    resumes_metadata = json.load(f)

class SearchRequest(BaseModel):
    query: str
    top_k: int = 10
    min_score: float = 0.0

@app.post("/search")
async def search_resumes(request: SearchRequest):
    # Create query embedding
    query_embedding = model.encode([request.query])
    query_embedding = query_embedding.astype('float32')
    faiss.normalize_L2(query_embedding)
    
    # Search
    scores, indices = index.search(query_embedding, request.top_k)
    
    # Format results
    results = []
    for idx, score in zip(indices[0], scores[0]):
        if score >= request.min_score:
            resume = resumes_metadata[idx]
            results.append({
                'filename': resume['filename'],
                'score': float(score),
                'file_path': resume['file_path']
            })
    
    return {'results': results}

@app.get("/health")
async def health():
    return {"status": "healthy", "num_resumes": len(resumes_metadata)}

# Run with: uvicorn api:app --reload
'''

with open('api.py', 'w') as f:
    f.write(api_code)

print("✓ API code saved to api.py")
print("\nTo run the API:")
print("  1. Install: pip install fastapi uvicorn")
print("  2. Run: uvicorn api:app --reload")
print("  3. Visit: http://localhost:8000/docs")

✓ API code saved to api.py

To run the API:
  1. Install: pip install fastapi uvicorn
  2. Run: uvicorn api:app --reload
  3. Visit: http://localhost:8000/docs


## Next Steps

### Immediate:
1. Test with real member resumes
2. Adjust `top_k` and `min_score` parameters
3. Integrate with your website

### Improvements:
1. **Add explicit skill matching**: Boost scores for exact skill matches
2. **Add filters**: GPA, graduation year, specific majors
3. **Fine-tune the model**: Use Reddit/Discord resumes as training data
4. **Hybrid search**: Combine semantic search with keyword matching
5. **Add reranking**: Use a cross-encoder for more accurate top results

### Advanced:
1. **Train custom model**: Fine-tune on your specific resume data
2. **Add LLM summarization**: Use GPT to summarize why each resume matches
3. **Build skill taxonomy**: Extract and normalize skills automatically
4. **Add analytics**: Track which resumes get the most recruiter views